# 🔧 題目 2：B2C 電商競品比價 — Solution

⚠️ 講師用。學員請用 `pipeline_starter.ipynb`。


## Section 0：環境設定


In [ ]:
import pandas as pd
import sqlite3
import os
import json
print('✅ 套件載入完成')


In [ ]:
OPENAI_API_KEY = ""
if os.path.exists(".env"):
    with open(".env") as f:
        for line in f:
            if line.startswith("OPENAI_API_KEY"):
                OPENAI_API_KEY = line.strip().split("=", 1)[1]
print("✅" if OPENAI_API_KEY else "⚠️ fallback")


---
## Section 1：Extract


### Step 1-1：讀取 CSV


In [ ]:
df_raw = pd.read_csv("products.csv")
print(f"📊 {len(df_raw)} 筆, {len(df_raw.columns)} 欄")
df_raw.head()


### Step 1-2：檢查


In [ ]:
print(df_raw.dtypes)
print("\n", df_raw.isnull().sum())
print("\n", df_raw.describe())


### Step 1-3：自由探索


In [ ]:
print(df_raw.iloc[:, 0].value_counts().head(10))
print(f"\n唯一值: {df_raw.iloc[:, 0].nunique()}")


### Step 1-4：SQLite


In [ ]:
conn = sqlite3.connect("pipeline.db")
df_raw.to_sql("raw_products", conn, if_exists="replace", index=False)
print(f"✅ raw_products: {pd.read_sql('SELECT COUNT(*) as n FROM raw_products', conn)['n'][0]} 筆")


---
## Section 2：Transform


### Step 2-1：從 raw 讀出


In [ ]:
df = pd.read_sql("SELECT * FROM raw_products", conn)
before = len(df)
print(f"讀出 {before} 筆")


### Step 2-2 ~ 2-4：清洗


In [ ]:
df = df.dropna(subset=["title", "amazon_price"])
df["amazon_price"] = pd.to_numeric(df["amazon_price"], errors="coerce")
df["flipkart_price"] = pd.to_numeric(df["flipkart_price"], errors="coerce")
df["amazon_rating"] = pd.to_numeric(df["amazon_rating"], errors="coerce")
df = df.dropna(subset=["amazon_price", "flipkart_price"])
df["flipkart_rating"] = pd.to_numeric(df["flipkart_rating"], errors="coerce").fillna(0)
df["amazon_rating"] = df["amazon_rating"].fillna(0)
df["price_diff"] = df["amazon_price"] - df["flipkart_price"]
df["price_diff_pct"] = ((df["price_diff"] / df["amazon_price"]) * 100).round(1)
df["cheaper_on"] = df.apply(lambda r: "Amazon" if r["price_diff"] < 0 else ("Flipkart" if r["price_diff"] > 0 else "Same"), axis=1)


In [ ]:
print(f"清洗前: {before} → 清洗後: {len(df)}")


### 🏁 檢查點


In [ ]:
assert df.isnull().sum().sum() == 0, "❌ 還有缺漏值"
assert "price_diff" in df.columns, "❌ 缺少 price_diff"
assert "cheaper_on" in df.columns, "❌ 缺少 cheaper_on"
print("✅ 通過")
print(f"   {len(df)} 筆, {len(df.columns)} 欄")


### Step 2-5：寫入 cleaned


In [ ]:
df.to_sql("cleaned_products", conn, if_exists="replace", index=False)
print(f"✅ cleaned_products")


---
## Section 3：SQL


### Step 3-1：價差最大的書


In [ ]:
price_gap = pd.read_sql("""
SELECT title, amazon_price, flipkart_price,
       ROUND(price_diff, 2) as price_diff, cheaper_on
FROM cleaned_products
ORDER BY ABS(price_diff) DESC
LIMIT 20
""", conn)
price_gap


### Step 3-2：各平台比較


In [ ]:
platform_stats = pd.read_sql("""
SELECT cheaper_on,
       COUNT(*) as book_count,
       ROUND(AVG(ABS(price_diff)), 2) as avg_diff
FROM cleaned_products
GROUP BY cheaper_on
ORDER BY book_count DESC
""", conn)
platform_stats


### Step 3-3：視覺化


In [ ]:
import matplotlib.pyplot as plt
# y 軸要用數值欄 price_diff（columns[-1] 是字串欄 cheaper_on，會畫不出來）
price_gap.head(10).plot.barh(x="title", y="price_diff", figsize=(10,5))
plt.tight_layout()
plt.show()

### Step 3-5：存結果


In [ ]:
os.makedirs("processed", exist_ok=True)
price_gap.to_csv("processed/price_gap.csv", index=False)
platform_stats.to_csv("processed/platform_stats.csv", index=False)
print("✅ 已存")


---
## Section 4：LLM


In [ ]:
import requests
def llm_analyze(text, api_key=None):
    if api_key: return _llm_api(text, api_key)
    return _llm_fallback(text)
def _llm_api(text, api_key):
    prompt = f"""請分析以下書籍資訊，回傳 JSON：
{{"category": "文學/科技/商業/教育/生活/其他", "insight": "一句話書籍分析"}}\n文字：{text[:300]}"""
    try:
        resp = requests.post("https://api.openai.com/v1/chat/completions",
            headers={"Authorization": f"Bearer {api_key}"},
            json={"model": "gpt-4o-mini", "messages": [{"role": "user", "content": prompt}], "temperature": 0.3}, timeout=30)
        content = resp.json()["choices"][0]["message"]["content"].strip()
          start = content.find("{"); end = content.rfind("}")
          if start != -1 and end != -1: content = content[start:end+1]
        return json.loads(content)
    except: return _llm_fallback(text)
def _llm_fallback(text):
    t = text.lower()
    if any(w in t for w in ["python","java","code","programming","data"]): cat = "科技"
    elif any(w in t for w in ["business","management","marketing","finance"]): cat = "商業"
    elif any(w in t for w in ["cook","health","fitness","travel","garden"]): cat = "生活"
    elif any(w in t for w in ["learn","study","guide","handbook","education"]): cat = "教育"
    elif any(w in t for w in ["novel","fiction","story","poetry","literature"]): cat = "文學"
    else: cat = "其他"
    return {"category": cat, "insight": text[:50] + "..."}
print("✅ LLM Helper")


### Step 4-1：單筆測試


In [ ]:
test = str(df["title"].iloc[0])
result = llm_analyze(test, OPENAI_API_KEY if OPENAI_API_KEY else None)
print(f"📝 {test[:60]}\n🤖 {result}")


### Step 4-2：批次


In [ ]:
BATCH_SIZE = 50
api_key = OPENAI_API_KEY if OPENAI_API_KEY else None
results = []
for i, row in df.head(BATCH_SIZE).iterrows():
    r = llm_analyze(str(row["title"]), api_key)
    results.append(r)
    if len(results) % 10 == 0: print(f"  {len(results)}/{BATCH_SIZE}")
print(f"✅ {len(results)} 筆")


### Step 4-3：整理 + 寫入


In [ ]:
df_analyzed = df.head(BATCH_SIZE).copy()
first_keys = list(results[0].keys())
for k in first_keys:
    df_analyzed[k] = [r.get(k, "") for r in results]
df_analyzed.rename(columns={k: "llm_insight" for k in first_keys if "insight" in k}, inplace=True)
df_analyzed.to_sql("analyzed_products", conn, if_exists="replace", index=False)
print("📊 三表：")
for t in ["raw_products", "cleaned_products", "analyzed_products"]:
    print(f"  {t}: {pd.read_sql(f'SELECT COUNT(*) as n FROM {t}', conn)['n'][0]}")


---
## Section 5：驗證


In [ ]:
lineage = pd.read_sql("""
SELECT \'raw_products\' as layer, COUNT(*) as rows FROM raw_products
UNION ALL SELECT \'cleaned_products\', COUNT(*) FROM cleaned_products
UNION ALL SELECT \'analyzed_products\', COUNT(*) FROM analyzed_products
""", conn)
print(lineage.to_string(index=False))


---
## Section 6：報告


In [ ]:
total_books = len(df)
avg_diff = df["price_diff"].mean()
platform_lines = "\n".join(
    f'- {r["cheaper_on"]}: {r["book_count"]} 本，平均價差 {r["avg_diff"]:.2f}'
    for _, r in platform_stats.iterrows()
)
report = f"""# 電商競品比價分析報告
## 資料概要
- 分析書籍：{total_books} 本
- 平均價差：{avg_diff:.2f}
## 平台比較
{platform_lines}
## 建議
1. 價差大的書優先調價
2. 定期追蹤競品定價變化
## Pipeline
CSV → pandas → SQLite → SQL → LLM → 本報告
"""
os.makedirs("output", exist_ok=True)
with open("output/pipeline_doc.md", "w") as f: f.write(report)
print("✅ pipeline_doc.md")


---
## Section 7：打包


In [ ]:
checks = [("pipeline.db","DB"), ("processed","統計"), ("output/pipeline_doc.md","報告")]
for p,d in checks: print(f"  {'✅' if os.path.exists(p) else '❌'} {d}: {p}")
c = sqlite3.connect("pipeline.db")
for t in ["raw_products","cleaned_products","analyzed_products"]:
    try: print(f"  ✅ {t}: {pd.read_sql(f'SELECT COUNT(*) as n FROM {t}', c)['n'][0]}")
    except: print(f"  ❌ {t}")
c.close()
